

## What we want our policy to achieve

The agents should ideally internalize trough the given rewards, the following concepts:
- safety distance: if I am approaching a car nearing distance = 0, I should slow down
- right of passage: i can occupy the crossing only if the time I take to clear it is shorter than the time it takes the other cars to reach it


## (1) We perform a Hyper Parameter Search (HPS) in a simple configuration 
destination/spawn config B  
initial vehicle = 1  
spawn rate = 0.6  
no speed limit  

RUN ./A-checkpoints/HPS_SimpleConfig_B_30exp_100iter_26-04-13

**Fixed params:**    
train_batch_size_per_learner=2048,  
minibatch_size=128,          
clip_param=0.1,  
lambda_ = 0.95,
vf_loss_coeff = 0.5,    
kl_target = 0.01,  

**Search space:**  
    num_epochs = tune.choice([5, 8, 10]),      
    lr = tune.loguniform(5e-6, 5e-5),  
    entropy_coeff = tune.loguniform(1e-4, 1e-2),   
    gamma = tune.choice([0.99, 0.995, 0.999]),



## (2) We identify 3 trials that stood out

**BEST** (best mean reward but unstable): ./A-checkpoints/HPS_SimpleConfig_B_30exp_100iter_26-04-13/RunID_0/PPO_Batch_2048-lr_6.964685361560775e-05_ID_91987465_BEST  
        entropy_coeff = 0,0005  
        num_epochs = 8  
        lr = 6.965e-05  
        gamma = 0.995  
    
**STABLE** (good stability): ./A-checkpoints/HPS_SimpleConfig_B_30exp_100iter_26-04-13/RunID_0/PPO_Batch_2048-lr_2.3754362565798656e-05_ID_735bc4cc_STABLE  
        entropy_coeff = 0,0028  
        num_epochs = 10  
        lr = 2.375e-05  
        gamma = 0.995  


**MORE STABLE** (great stability, wose reward): ./A-checkpoints/HPS_SimpleConfig_B_30exp_100iter_26-04-13/RunID_0/PPO_Batch_2048-lr_8239e-06_ID_fc0cd4b9_MORE_STABLE  
        entropy_coeff = 0.00234  
        num_epochs = 8  
        lr = 8.2397e-06  
        gamma = 0.995  


we found the best combination of HP for better results and stability to be the parameters of the STABLE trial  
        entropy_coeff = 0,0028  
        num_epochs = 10   
        lr = 2.375e-05  
        gamma = 0.995  



## (4) We trained for 500 iterations Simple Config B with the optimal HP found in the HPS,
In the training two agents face spawn point and destination according to 'config B':  
- No speed limit was set.  
- the observation were absolute ABSOLUTE = True (not relative).  
- Spawn rate = 0.6   
- Initial Vehicle = 1  

entropy_coeff = 0,0028  
num_epochs = 10  
lr = 2.375e-05  
gamma = 0.99  

RUN ./A-checkpoints/run4/SimpleConfig_B_500iter


<img src="./Graphs/ZSG/ZSG_run4.svg" alt="ZSG graph" width="1200" height="900">

    Note:
        Alta varianza nei risultati
        buona generalizzazione in configurazioni "Speculari"
        mostra generalizzazione in configurazioni diverse, ma non risolve a pieno l'ambiente
        chiari segni di Overfitting
        esmpio: se la destinazione = uscita Nord (mai vista in training), l'agente spesso si ferma pochi metri prima del traguardo



## (5) We recreate the previous training with a few important differences and added difficulties
**Reward and Observation:**  
- Absolute is set to False, (we hope this will increase generalization as the agent will learn the intersection from its own perspective instead of a "overview" vision)
- The reward_speed_range is changed to [0,6] (we hope this will incentivize agents to take more time when facing the enviroment)  

**Environment changes:**    
- The initial vehicle count is upped to 4, 
- spawn_rate is lowered to 0.2, 
- the episode starts with the initial vehicles already crossing the intersection (initial_simulation_steps = 7)
- a speed limit of 6 is set in the intersection,

the idea is to start the episode with an obstacle in the middle, or else the agents will learn to clear the intersection as fast as possible before any other vehicle can reach it.
   

RUN ./A-checkpoints/run5/PPO_1

ZSG on checkpoint 49:  

<img src="./Graphs/ZSG/ZSG_run5onImprovedConfig.svg" alt="ZSG graph" width="1200" height="900">

We now evaluate the policy on the same spawn rates as run4:

<img src="./Graphs/ZSG/ZSG_run5onSimpleConfig.svg" alt="ZSG graph" width="1200" height="900">

The resulting policy appears to better generalize when we change the agents destinations and positions.   
Some behaviours need fixing ie:   
agents sometimes let eachoter pass, other times they rush full speed and collide, this could be caused by randomization in the spawning distance of the agents.   
Agents still ram into non controlled vehicles when they are in the middle of the intersection even if staying still would be the best course of action.  
The increased difficulty tanked performace.



# (6) To address these issues we update:   
**"target_speeds": [0, 1.5, 3, 4.5, 6]**  
This way the agents can choose a lower speed to reach the intersection and have more time to make a decision.  
**"high_speed_reward":0.5**  
we want to limit the importance of reaching the maximum speed while still incentivizing movement
**"collision_reward": -200**.  
collisions must be avoided at all costs, even if it means not reaching the goal  
**"duration": 60**.  

we increase duration because we want to allow the agents to have the time to wait if the intersection is full 
**lr = 1e-4**  
we increase the learning rate as it looks like improvement platoes at around 200 iter

RUN: ./A-checkpoints/run6/PPO_0

This run produced terrible results, 
with such a collision reward, one of the agents learned to stay still most of the time, instead of attempting the intersection



## (7) we return to a collision reward of: -50, while keeping the other updates, we also add a scheduled lr that diminishes over time
RUN: ./A-checkpoints/run7/PPO_5

- "target_speeds": [0, 1.5, 3, 4.5, 6]
- "high_speed_reward":0.5
- "duration":60  
- lr =   [ [0, 5e-05],[500000,1e-05],[1000000,1e-06]  ]


depending on the given spawn points and destinations, behaviour varies widely:
on config A the agents stand still, resulting in all the episodes being truncated
on config C the agents solve the intersection with 80% success rate

when a vehicle spawns on 0, the policy learned to stand still for some reason

further training to iter 770 decreased std deviation but didnt imporve generalization

ZSG on checkpoint 52:  

<img src="./Graphs/ZSG/ZSG_run7onImprovedConfig.svg" alt="ZSG graph" width="1200" height="900">


We now evaluate the policy on the same spawn rates as run 4, for a valid comparison:  

<img src="./Graphs/ZSG/ZSG_run7onSimpleConfig.svg" alt="ZSG graph" width="1200" height="900">



## (8) a penalty for being stopped and going backwards is introduced, rewards are normalized

**to address the Freezed agents from run7, we introduce a handful of penalties that encourage the agents to progress trough the episode and avoid collisions:**

- "ego_vehicle_speed_limit": 9, 
- "speeding_penalty": -0.5,   activates whe speed > "ego_vechicle_speed_limit"
- "stopped_penalty": -0.5,     activated when speed <= 0
- "tailgating_penalty": -2,   

the tailgating penalty activates when a ego vehicles approaches the vehicle ahead, this incentivizes the agent to maintain a safety distance:  

 
```python 
 if front_vehicle is not None:
 distance = np.linalg.norm(vehicle.position - front_vehicle.position)
                
                if distance < 15.0:
                    relative_speed = vehicle.speed - front_vehicle.speed
                    if relative_speed > 0: 
                 
                        danger_factor = (15.0 - distance) / 15.0 
                        tailgating_signal = danger_factor
```
               
the danger factor is then multiplied according to "tailgating_penalty".  

the idea is to give the ppo algo a hint as to where the issue is before a crash, the algo now sees:  
ok, 3 steps before the crash, the tailgating penalty started increasing, I need to change my actions if that penalty increases.
  
  
  
  
**To increase stability in training, with the aim of stabilizing the Value Function Loss and Policy Entropy: we update the hyperparameters as follows:**

```python 
train_batch_size_per_learner=4096,   #increased from 2048
minibatch_size=256,          #increased from 128
clip_param=0.1,                


entropy_coeff = 0.0028,             
num_epochs = 4,                     #lowered to avoid overfitting on bigger batch size
lr = [[0, 1e-5], [1000000, 1e-6]], #updated to be a slower, smoother transition

    
#policy and value function dont share weights, each has its own NN
model={
"vf_share_layers": False,
}            

#unchanged
gamma = 0.995,   

use_critic = True,           
use_gae = True,     
lambda_ = 0.95,
vf_loss_coeff = 0.5,   
kl_target = 0.01,     

```

In our hyperparameter configuration, we chose to separate the Policy (Actor) and Value Function (Critic) by setting "vf_share_layers": False.  
This architectural decision ensures that the two distinct optimization objectives do not interfere with one another.  
The Actor's task is a probability distribution problem (selecting the optimal discrete action),  
while the Critic performs a regression task (predicting the exact expected return V(st)).  

**Rewards are now normalized in a range of [-1,1] where -1 is equal to a crash (-50) and 1 is equal to arriving at destination (+50)**


The results for Zero Shot Generalization show great generalization for the Horizontal axis.
The policy struggles to obtain successes when the spawn points are on the Vertical axis (never seen in training).

Initially, we suspected this was caused by the "observe_intentions": True attribute,  
which adds an absolute one-hot encoding of the vehicles' intended destinations (e.g., North, South).  
However, further debugging revealed a deeper structural issue:  
the environment was still feeding the agent absolute global coordinates instead of relative ones.

The following graph shows the ZSG for policy 8, using absolute observations:

<img src="./Graphs/ZSG/ZSG_run8.svg" alt="ZSG graph" width="1200" height="900">



# BUG 

We noticed a critical bug in the library where setting Observations -> absolute = False
did not actually change the observations to relative.

We suspect this is due to the MultiAgentWrapper from the ray rllib library.

To fix this, we force observations to be relative in our MA_wrapper by subtracting to every vehicle in the observation matrix (incuding the ego vehicle), the ego_vehicles position and speed.
The change is implemented by the following function in MA_wrapper.py:

```python 
    def _process_obs(self, agent_obs_matrix):

        if self._is_absolute:
            return agent_obs_matrix.flatten().astype(np.float32)

       
        rel_obs = agent_obs_matrix.copy()
        # 2. extract [x, y, vx, vy] from ego vehicle
        ego_vals = rel_obs[0, 1:5].copy() 
        # True only where the vehicle is present (presence = 1)
        present_mask = rel_obs[:, 0] == 1.0
        # subtract only where a vehicle is present (presence = 1)
        rel_obs[present_mask, 1:5] -= ego_vals
        
        return rel_obs.flatten().astype(np.float32)
```



For now, observation features regarding the Steering angle remain absolute (0° is the same direction for every agent at any time)

## (9) Relative Observations are now actually introduced 

RUN: ./A-checkpoints/run9

We notice an issue with a key metric
while the vf loss is steadily decreasing,
the vf explained var is struggling at predicting the variance

To improve this metric we apply the following changes:

Now that observations are relative, we utilize sparser rewards, by removing penalties that guided the agents  
"speeding_penalty": -0.01,     #change from -2  
"tailgating_penalty": 0,   #disabled, changed from -2  
"stopped_penalty": -0.05,   #lowered from -0.5  
   
The gamma currently selected might be excessive  
gamma = 0.995 (has a horizon of over 200 steps, our max steps nr is 60!!)  

we will explore different gammas with:  
    gamma = tune.grid_search([0.95, 0.98, 0.99])  
and find the optimal

we increase vf_loss_coeff = 1,    # changed from 0.5  
This increases the weight of the Value Function loss in the total PPO objective,   
effectively forcing the optimizer to prioritize the Critic's accuracy, which is crucial since the Actor and Critic now use separate NNs.  

## (10) We apply the changes mentioned in run 9 and perform exploration of gammas

RUN: ./A-checkpoints/run10/PPO_0

 
**the superiority of gamma = 0.95**  
Comparing the performance across the different graphs, it is evident that regardless of the discount factor, the algorithm reliably converges to an approximate 80% success rate.    
However, γ significantly influences the agent's urgency to clear the intersection.    
The trial with γ=0.95 successfully resolves the scenario in an average of 24 steps, whereas the other configurations take significantly more time.    
Since our environment is truncated at a maximum of 60 steps, a high γ introduces excessive noise into the Value Function by evaluating overly long temporal horizons.    
Furthermore, we observe that critical diagnostic metrics such as vf_explained_var, vf_loss, and entropy—all perform noticeably better with γ=0.95.  
This demonstrates that the Critic learns the environment's dynamics much faster, allowing the agent to be more confident and efficient in its predictions.  
-  results between the different gammas:  

<img src="./Graphs/Tensorboard/gammaSearch-vfLoss.png" alt="ZSG graph" width="1200" height="350">

<img src="./Graphs/Tensorboard/gammaSearch-numSteps.png" alt="ZSG graph" width="900" height="300">

<img src="./Graphs/Tensorboard/gammaSearch-Success.png" alt="ZSG graph" width="900" height="300">



**Run 9 vs. Run10**  
Comparing Run 9 with the adjusted hyperparameters of Run 10 (γ=0.95, vf_loss_coeff = 1.0, and sparser rewards), we observe a textbook example of the exploration-exploitation trade-off induced by reward shaping.  
By removing dense guiding penalties such as speeding and tailgating the agent initially takes longer to discover safe, goal oriented behaviors.  
This is evident in the slower initial ascent of the success rate and the delayed drop in the crash rate during the first 30 iterations.

However, once the policy converges, thet policy from run10 matches the baseline's efficiency and introduces a significant behavioral improvement: it completes the episodes much faster.  
The reduction of the discount factor to γ=0.95 successfully instilled a stronger sense of urgency, lowering the average episode length from roughly 32 steps to approximately 24 steps.  
Furthermore, the combination of a more realistic temporal horizon and an increased vf_loss_coeff resulted in a much healthier Critic.  
This is demonstrated by a tighter Value Function Loss, a smoother entropy decay, and a visibly higher and more stable Explained Variance, confirming that the Critic is now evaluating the environment's states with greater accuracy and confidence.



**addressing low vf_explained_var**  
We can obseerve in the graphs, that the vf_loss metric correctly decreases and slowly stabilizes around 0.1, this indicates a healthy critic.
However, the vf_explained_var metric appears to be unusually low (0.1-0.3): considering the high reward, success rate and the low loss, this metric should ideally grow to aproximately 0.6.  

Value Function Explained Var formula:
$$
1 - \frac{Var(R_t - V(s_t))}{Var(R_t)}
$$

The low vf_loss, high reward and success rate, prove that the predictions of the Generalized Advantage Estimation (GAE) are guiding the Policy (Actor) in the right direction.

using raw unnormalized rewards with high variance leads to severe network instability and an even worse vf_explained_var:  

Normalized rewards:  
<img src="./Graphs/Tensorboard/explained-var-Normalized.png" alt="ZSG graph" width="900" height="300">  
Raw rewards:  
<img src="./Graphs/Tensorboard/explained-var-Raw.png" alt="ZSG graph" width="900" height="300">  




## 11. Zero-Shot Generalization (ZSG) on Run 10 Baseline Config (500 Iterations)

First, we evaluate the Zero-Shot Generalization capabilities of the baseline policy from Run 10, trained for 500 iterations. Obtaining the following results:

<img src="./Graphs/ZSG/run11-SpatialRelativity.svg" alt="ZSG graph" width="1200" height="900">

As observed, the policy still significantly underperforms in environments involving the vertical axis. 

To investigate and address this, we applied an **Ego-Centric Rotational Invariance** transformation to the observation space (refer to the `_process_obs` function in `MA_wrapper.py`).  
We then re-evaluated the exact same policy using these newly rotated observations:

<img src="./Graphs/ZSG/run11-RotationalInvariance.svg" alt="ZSG graph" width="1200" height="900">

**Analysis of the Results:**
It is crucial to highlight that we are utilizing a policy originally trained on absolute, global map orientations.  
However, the rotational transformation acts as a geometric filter: when the agent is spawned on a vertical axis, the wrapper applies an inverse rotation matrix.   
As a result, the agent perceives its heading features (cosine and sine) as if it were constantly driving East, perfectly matching its training conditions.  
By artificially projecting the world into this familiar reference frame, the agent avoids Out-of-Distribution (OOD) panic and successfully generalizes its driving behavior regardless of the global spawn location.

## The Limits of Translational Invariance and the Need for Rotational Invariance

While calculating relative positions successfully enforces translational invariance, the evaluation metrics demonstrate that the policy still fails to generalize in scenarios where the Ego vehicle originates from the vertical axis.  
This vulnerability stems from the heading features (cos(θ),sin(θ)) and velocity vectors (vx,vy), which remain anchored to the global coordinate system.

Because the training environment predominantly exposes the agent to horizontal traffic flow (heading angles of 0∘ or 180∘), 
the Multi-Layer Perceptron (MLP) overfits to these specific feature activations.  
When deployed zero-shot on the vertical axis (90∘ or 270∘), 
the network receives out-of-distribution inputs for the heading and velocity dimensions, leading to catastrophic failure.

To achieve complete spatial generalization, two approaches can be considered:  

- Data-Driven Approach (Domain Randomization): Expanding the training distribution by mixing horizontal and vertical start configurations. This forces the neural network to develop robust weights for all possible heading states.

- Mathematical Approach (Rotational Invariance): Implementing a 2D rotation matrix within the environment wrapper. By rotating the relative (x,y) coordinates and (vx,vy) velocities by the inverse of the Ego vehicle's heading angle, the observation space is transformed into an ego-centric frame where the agent mathematically perceives itself to be continuously moving straight ahead, effectively neutralizing the domain shift.



## 12. Training Dynamics and ZSG with Ego-Centric Observations

Next, we train a new policy from scratch exclusively using the updated observations with Rotational Invariance applied. 

Comparing the training metrics with the baseline (Run 11), we observe a temporary decrease in initial performance:

<img src="./Graphs/Tensorboard/run11vsrun12_vf.png" alt="ZSG graph" width="1600" height="400">
<img src="./Graphs/Tensorboard/run11vsrun12_succ.png" alt="ZSG graph" width="1600" height="400">


We observed the entropy metric dropping too quickly below 1.0 (indicating premature convergence). To counter this and maintain healthy exploration, we tuned the PPO algorithm by increasing the `entropy_coeff` parameter.

**Zero-Shot Generalization:**
Applying the ZSG test to this newly trained policy (Checkpoint 09), we obtained the following results:

<img src="./Graphs/ZSG/run12-checkpoint09.svg" alt="ZSG graph" width="1200" height="900">